In [1]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

#!wget {PREFIX}/01-agentic-rag/code/ingest.py
#!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
#!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

In [2]:
## Load the FAQ data
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
## Generate questions for LLM Zoomcamp course:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [4]:
# use these docs - reassign variable name
documents = documents_llm


In [5]:
# each document id has both - question and answer
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
## Structure output with pydantic
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


In [7]:
## LLM Instruction:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
## call LLM for 1 document:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
## shape the document as json
import json

user_prompt = json.dumps(doc)

In [10]:
## create the messages:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]


Until now we called `responses.create` and read `response.output_text`. 
<br> For structured output we switch to `responses.parse` and pass `text_format=Questions`, which tells the API to return our class instead of free text.

In [11]:
## call the model:

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['I just found out about this course — can I still sign up and take it?', 'Is it too late to join the course now, or can new students still come in?', 'If I start the course late, can I still get the certificate somehow?', 'Do I need to finish and submit the project before submissions close to be eligible for a certificate?', 'Can I participate in the course after it has already started, or is that not allowed?']


In [13]:
# Directly accessing the questions
for question in result.questions:
    print(question)

I just found out about this course — can I still sign up and take it?
Is it too late to join the course now, or can new students still come in?
If I start the course late, can I still get the certificate somehow?
Do I need to finish and submit the project before submissions close to be eligible for a certificate?
Can I participate in the course after it has already started, or is that not allowed?


In [14]:
## let's put everyting together as a reusable utilities helper

from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [16]:
for question in result.questions:
    print(question)

Can I still join the course if I found it late?
Is it too late to start this course now?
If I join after the course has started, can I still get a certificate?
What do I need to do to receive the certificate if I’m joining late?
Are late submissions accepted for the final project so I can still qualify for a certificate?


In [17]:
## Tracking costs:
usage.input_tokens, usage.output_tokens

(207, 87)

In [18]:
## import price helper
from evaluation_utils import calc_price

In [19]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00039150000000000003,
 'total_cost': 0.00054675}

In [20]:
# convert the questions into the ground truth records:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start this course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to receive the certificate if I’m joining late?',
  'document': '74eb249bbf'},
 {'question': 'Are late submissions accepted for the final project so I can still qualify for a certificate?',
  'document': '74eb249bbf'}]

In [21]:
## For each document, we generate questions and save them as ground truth records.
from evaluation_utils import llm_structured_retry

In [22]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [23]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [24]:
## Parallel processing
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [25]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [ ]:
## split ground truth and usage into 2 parts:

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

565

In [ ]:
# roughly 5 times more questions that the documents (answers)
len(ground_truth), len(documents)

(565, 113)

In [29]:
ground_truth[0]

{'question': 'I just found this course — am I still allowed to join, or is it too late?',
 'document': '74eb249bbf'}

In [36]:
ground_truth[1]

{'question': 'If I join now, can I still get a certificate somehow?',
 'document': '74eb249bbf'}

In [35]:
documents[0]['id']

'74eb249bbf'

In [37]:
## Calculate usage:

from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08688749999999998

In [38]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08688749999999998

In [39]:
#Create a dataframe so we can look at the records as a table and save them as a CSV file.
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)


In [41]:
df_ground_truth.head(7)

,question,document
0,I just found this course — am I still allowed ...,74eb249bbf
1,"If I join now, can I still get a certificate s...",74eb249bbf
2,What do I need to do to qualify for the certif...,74eb249bbf
3,Is there a deadline for submitting the project...,74eb249bbf
4,Can I enroll after the course has already star...,74eb249bbf
5,Do I actually need a confirmation email to joi...,977bf7786c
6,"If I registered for the course, do I have to w...",977bf7786c


In [44]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)